In [68]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from datetime import date

In [13]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [14]:
filepath = "../data/yellow_tripdata_2025-11.parquet"

In [15]:
df = spark.read \
    .option("header", "true") \
    .option("infereSchema","true") \
    .parquet(filepath)

In [16]:
df.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double'),
 ('cbd_congestion_fee', 'double')]

In [17]:
df = df.repartition(4)

In [ ]:
df.write.parquet("../data/spark_partititoned/")

In [49]:
df = spark.read.parquet("../data/spark_partititoned/")

In [45]:
df.filter(F.to_date("tpep_pickup_datetime") == F.to_date(F.lit("2025-11-15")))\
    .count()

162604

In [ ]:
df = df.withColumn(
    "trip_hours_diff",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

In [55]:
df.select(
    F.max("trip_hours_diff")
).show()

+--------------------+
|max(trip_hours_diff)|
+--------------------+
|   90.64666666666666|
+--------------------+



In [66]:
zones_df = spark.read.option("infereSchema","true").csv(path="../data/taxi_zone_lookup.csv", header=True)

In [69]:
zones_df = zones_df.withColumn("LocationID", zones_df.LocationID.cast(T.IntegerType()))

In [70]:
zones_df.dtypes

[('LocationID', 'int'),
 ('Borough', 'string'),
 ('Zone', 'string'),
 ('service_zone', 'string')]

In [71]:
df = df.join(zones_df, df.PULocationID == zones_df.LocationID)

In [74]:
zones_df.head(5)

[Row(LocationID=1, Borough='EWR', Zone='Newark Airport', service_zone='EWR'),
 Row(LocationID=2, Borough='Queens', Zone='Jamaica Bay', service_zone='Boro Zone'),
 Row(LocationID=3, Borough='Bronx', Zone='Allerton/Pelham Gardens', service_zone='Boro Zone'),
 Row(LocationID=4, Borough='Manhattan', Zone='Alphabet City', service_zone='Yellow Zone'),
 Row(LocationID=5, Borough='Staten Island', Zone='Arden Heights', service_zone='Boro Zone')]

In [75]:
df.groupBy("Zone") \
    .count() \
    .orderBy("count") \
    .show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
|       Rikers Island|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|       West Brighton|   14|
|New Dorp/Midland ...|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
